In [1]:
# =============================================================
# Modul 0: Datenvorbereitung / Data Preparation
# Projekt: RetailCo Financial Analysis
# Datensatz: Bike Sales in Europe (Kaggle, Sadiq Shah)
# Autor: Yurii Oleshchuk
# Datum: 2026
# =============================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# 1. DATEN LADEN / LOAD DATA
# DE: Rohdaten aus dem CSV-Datei laden
# EN: Load raw data from CSV file
# ------------------------------------------------------------------

df = pd.read_csv('../data/raw/bike_sales.csv', encoding='latin1')

print("Shape:", df.shape)
print(df.dtypes)
print(df.head(3))

Shape: (113036, 18)
Date                  str
Day                 int64
Month                 str
Year                int64
Customer_Age        int64
Age_Group             str
Customer_Gender       str
Country               str
State                 str
Product_Category      str
Sub_Category          str
Product               str
Order_Quantity      int64
Unit_Cost           int64
Unit_Price          int64
Profit              int64
Cost                int64
Revenue             int64
dtype: object
         Date  Day     Month  Year  Customer_Age       Age_Group  \
0  11/26/2013   26  November  2013            19     Youth (<25)   
1  11/26/2015   26  November  2015            19     Youth (<25)   
2   3/23/2014   23     March  2014            49  Adults (35-64)   

  Customer_Gender    Country             State Product_Category Sub_Category  \
0               M     Canada  British Columbia      Accessories   Bike Racks   
1               M     Canada  British Columbia      Accessories  

In [2]:
# ------------------------------------------------------------------
# 2. DATENQUALITÄT PRÜFEN / DATA QUALITY CHECK
# DE: Fehlende Werte, Duplikate und Ausreißer identifizieren
# EN: Identify missing values, duplicates and outliers
# ------------------------------------------------------------------

print("=== Fehlende Werte / Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplikate / Duplicates ===")
print(f"Duplikate gefunden: {df.duplicated().sum()}")

print("\n=== Deskriptive Statistik / Descriptive Statistics ===")
print(df[['Revenue', 'Cost', 'Profit', 'Order_Quantity', 
          'Unit_Price', 'Unit_Cost']].describe().round(2))

=== Fehlende Werte / Missing Values ===
Date                0
Day                 0
Month               0
Year                0
Customer_Age        0
Age_Group           0
Customer_Gender     0
Country             0
State               0
Product_Category    0
Sub_Category        0
Product             0
Order_Quantity      0
Unit_Cost           0
Unit_Price          0
Profit              0
Cost                0
Revenue             0
dtype: int64

=== Duplikate / Duplicates ===
Duplikate gefunden: 1000

=== Deskriptive Statistik / Descriptive Statistics ===
         Revenue       Cost     Profit  Order_Quantity  Unit_Price  Unit_Cost
count  113036.00  113036.00  113036.00       113036.00   113036.00  113036.00
mean      754.37     469.32     285.05           11.90      452.94     267.30
std      1309.09     884.87     453.89            9.56      922.07     549.84
min         2.00       1.00     -30.00            1.00        2.00       1.00
25%        63.00      28.00      29.00          

In [3]:
# ------------------------------------------------------------------
# 3. DATENTRANSFORMATION / DATA TRANSFORMATION
# DE: Spalten umbenennen, Datentypen korrigieren, neue Spalten anlegen
# EN: Rename columns, fix data types, create new columns
# ------------------------------------------------------------------

# Spaltennamen vereinheitlichen / Standardize column names
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Datum parsen / Parse date
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

# Neue Spalten / New columns
df['year_month']    = df['date'].dt.to_period('M')   # für Monatsabschluss
df['quarter']       = df['date'].dt.quarter.map(lambda q: f'Q{q}')
df['fiscal_year']   = df['date'].dt.year

# Marge berechnen / Calculate margin
df['gross_margin_pct'] = (df['profit'] / df['revenue'] * 100).round(2)

# Prüfen / Check
print(df[['date','year_month','quarter','fiscal_year',
          'revenue','cost','profit','gross_margin_pct']].head(5))

        date year_month quarter  fiscal_year  revenue  cost  profit  \
0 2013-11-26    2013-11      Q4         2013      950   360     590   
1 2015-11-26    2015-11      Q4         2015      950   360     590   
2 2014-03-23    2014-03      Q1         2014     2401  1035    1366   
3 2016-03-23    2016-03      Q1         2016     2088   900    1188   
4 2014-05-15    2014-05      Q2         2014      418   180     238   

   gross_margin_pct  
0             62.11  
1             62.11  
2             56.89  
3             56.90  
4             56.94  


In [4]:
# ------------------------------------------------------------------
# 4. LÄNDER-MAPPING (DE + EN)
# DE: Ländernamen auf Deutsch und Englisch hinzufügen
# EN: Add country names in German and English
# ------------------------------------------------------------------

country_map_de = {
    'Australia':      'Australien',
    'Canada':         'Kanada',
    'France':         'Frankreich',
    'Germany':        'Deutschland',
    'United Kingdom': 'Vereinigtes Königreich',
    'United States':  'Vereinigte Staaten'
}

df['land_de'] = df['country'].map(country_map_de)  # Deutsch
df['land_en'] = df['country']                       # English (Original)

print(df[['country','land_de']].value_counts())

country         land_de               
United States   Vereinigte Staaten        39206
Australia       Australien                23936
Canada          Kanada                    14178
United Kingdom  Vereinigtes Königreich    13620
Germany         Deutschland               11098
France          Frankreich                10998
Name: count, dtype: int64


In [5]:
# ------------------------------------------------------------------
# 5. BUDGET-SPALTEN GENERIEREN / GENERATE BUDGET COLUMNS
# DE: Plan-Werte simulieren (±12% Abweichung vom Ist-Wert)
#     → Standard-Methode im Controlling für historische Projekte
# EN: Simulate plan values (±12% deviation from actual)
#     → Standard method in Controlling for historical datasets
# ------------------------------------------------------------------

np.random.seed(42)  # Reproduzierbarkeit / Reproducibility

n = len(df)
variation = np.random.uniform(-0.12, 0.12, n)

df['revenue_plan'] = (df['revenue'] * (1 + variation)).round(0)
df['cost_plan']    = (df['cost']    * (1 + variation * 0.8)).round(0)
df['profit_plan']  = (df['revenue_plan'] - df['cost_plan']).round(0)

# Abweichungen / Variances
df['revenue_var_abs'] = df['revenue'] - df['revenue_plan']
df['revenue_var_pct'] = (df['revenue_var_abs'] / df['revenue_plan'] * 100).round(2)

print("Budget-Spalten erstellt / Budget columns created:")
print(df[['revenue','revenue_plan','revenue_var_abs','revenue_var_pct']].head(5))

Budget-Spalten erstellt / Budget columns created:
   revenue  revenue_plan  revenue_var_abs  revenue_var_pct
0      950         921.0             29.0             3.15
1      950        1053.0           -103.0            -9.78
2     2401        2535.0           -134.0            -5.29
3     2088        2137.0            -49.0            -2.29
4      418         383.0             35.0             9.14


In [6]:
# ------------------------------------------------------------------
# 6. BEREINIGTE DATEN SPEICHERN / SAVE CLEANED DATA
# DE: Verarbeitete Daten als CSV speichern
# EN: Save processed data as CSV
# ------------------------------------------------------------------

df.to_csv('../data/processed/bike_sales_clean.csv', index=False, encoding='utf-8-sig')

print(f"✓ Datei gespeichert / File saved: bike_sales_clean.csv")
print(f"✓ Zeilen / Rows: {len(df):,}")
print(f"✓ Spalten / Columns: {len(df.columns)}")
print(f"\nFinale Spalten / Final columns:")
print(df.columns.tolist())

✓ Datei gespeichert / File saved: bike_sales_clean.csv
✓ Zeilen / Rows: 113,036
✓ Spalten / Columns: 29

Finale Spalten / Final columns:
['date', 'day', 'month', 'year', 'customer_age', 'age_group', 'customer_gender', 'country', 'state', 'product_category', 'sub_category', 'product', 'order_quantity', 'unit_cost', 'unit_price', 'profit', 'cost', 'revenue', 'year_month', 'quarter', 'fiscal_year', 'gross_margin_pct', 'land_de', 'land_en', 'revenue_plan', 'cost_plan', 'profit_plan', 'revenue_var_abs', 'revenue_var_pct']
